In [4]:
#IMPORTS

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader


import pandas as pd

from transformers import BertTokenizer, BertModel

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

import joblib as jl

c:\Users\Evan\anaconda3\envs\PhishGPU\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# MOUNTING DRIVE AND CHECKING COLAB CONNECTION

if not torch.cuda.is_available():
    from google.colab import drive
    drive.mount('/content/drive')
    %ls
    !nvidia-smi

In [6]:
# BERT CLASSIFIER DEFINITION

class BertClassifier(nn.Module):

    def __init__(self):
        super().__init__()

        self.bert = BertModel.from_pretrained('bert-base-uncased')

        #for param in self.bert.parameters():
            #param.requires_grad = False

        self.dropout = nn.Dropout(0.1)
        self.fc = nn.Linear(768, 1)

    def forward(self, input_ids, attention_mask):

        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        cls_embedding = outputs.last_hidden_state[:, 0, :]

        x = self.dropout(cls_embedding)

        x = self.fc(x)

        return x

In [7]:
#DATASET

#No Colab vs Colab Version
if torch.cuda.is_available():
    df = pd.read_csv("../../assets/cleaned_data/scikit_cleaned.csv")
else:
    df = pd.read_csv("/content/drive/MyDrive/SevenPhishingEmails/scikit_cleaned.csv")

texts = (
    df['sender'].fillna('') + ' ' +
    df['receiver'].fillna('') + ' ' +
    df['date'].fillna('') + ' ' +
    df['subject'].fillna('') + ' ' +
    df['body'].fillna('')
)

labels = df['label']

In [8]:
# 60-20-20 Split

X_train, X_temp, y_train, y_temp = train_test_split(
    texts,
    labels,
    test_size=0.4,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    random_state=42
)

In [9]:
# TOKENIZER

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

In [10]:
#PRE-ENCODING

encodings = tokenizer(
    texts.tolist(),
    truncation=True,
    padding=True,
    max_length=256,
    return_tensors='pt'
)

train_encodings = tokenizer(
    list(X_train),
    truncation=True,
    padding=True,
    max_length=256,
    return_tensors="pt"
)

val_encodings = tokenizer(
    list(X_val),
    truncation=True,
    padding=True,
    max_length=256,
    return_tensors="pt"
)

test_encodings = tokenizer(
    list(X_test),
    truncation=True,
    padding=True,
    max_length=256,
    return_tensors="pt"
)

In [11]:
# DATASET CLASS

class EmailDataset(Dataset):

    def __init__(self, encodings, labels):

        self.encodings = encodings
        self.labels = labels.tolist()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):

        return {
            'input_ids': self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'label': torch.tensor(self.labels[idx], dtype=torch.float32)
        }

In [12]:
# DATALOADERS

train_dataset = EmailDataset(train_encodings, y_train)
val_dataset = EmailDataset(val_encodings, y_val)
test_dataset = EmailDataset(test_encodings, y_test)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8)
test_loader = DataLoader(test_dataset, batch_size=8)

In [13]:
# MODEL, OPTIMIZER, LOSS FUNCTION

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = BertClassifier().to(device)

optimizer = optim.AdamW(model.parameters(), lr=2e-5)

criterion = nn.BCEWithLogitsLoss()

Using device: cuda


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4666.85it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [14]:
# TRAINING LOOP

from numpy import rint


EPOCHS = 3

for epoch in range(EPOCHS):
    model.train()

    print(f"training! Epoch {epoch+1}/{EPOCHS}")

    total_loss = 0

    for batch in train_loader:

        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)

        attention_mask = batch['attention_mask'].to(device)

        labels = batch['label'].to(device)

        outputs = model(input_ids, attention_mask).squeeze(1)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} Loss: {total_loss:.4f}")

training! Epoch 1/3
Epoch 1 Loss: 256.3864
training! Epoch 2/3
Epoch 2 Loss: 75.1757
training! Epoch 3/3
Epoch 3 Loss: 43.1492


In [15]:
# VALIDATION

model.eval()

predictions = []
actuals = []

with torch.no_grad():

    for batch in val_loader:

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)

        outputs = model(input_ids, attention_mask)

        probs = torch.sigmoid(outputs)

        preds = (probs >= 0.5).int().cpu().numpy()

        predictions.extend(preds.flatten())

        actuals.extend(batch['label'].numpy())

val_acc = accuracy_score(actuals, predictions)

print("Validation Accuracy:", val_acc)

Validation Accuracy: 0.9930403402500322


In [16]:
# FINAL TESTING AND SAVING 

def final_test():
    model.eval()

    predictions = []
    actuals = []

    with torch.no_grad():

        for batch in val_loader:

            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)

            outputs = model(input_ids, attention_mask)

            probs = torch.sigmoid(outputs)

            preds = (probs >= 0.5).int().cpu().numpy()

            predictions.extend(preds.flatten())

            actuals.extend(batch['label'].numpy())

    test_acc = accuracy_score(actuals, predictions)

    print("Test Accuracy:", test_acc)
    print(confusion_matrix(actuals, predictions))
    print(classification_report(actuals, predictions))

    metadata = {
        "validation_accuracy": val_acc,
        "test_accuracy": test_acc
    }

    #No Colab vs Colab version
    if(torch.cuda.is_available()):
        torch.save(model.state_dict(), "../../assets/models/bert_model.pth")
        torch.save(metadata, "../../assets/metadata/bert_metadata.pth")
        tokenizer.save_pretrained("../../assets/tokenizers/bert_tokenizer")
    else:
        torch.save(model.state_dict(), "/content/drive/MyDrive/PhishingModels/bert_model.pth")
        torch.save(metadata, "/content/drive/MyDrive/PhishingModels/bert_metadata.pth")
        tokenizer.save_pretrained("/content/drive/MyDrive/PhishingModels/bert_tokenizer")

    

 

    

In [17]:
#again commented so don't accidentally run

#final_test() 

Test Accuracy: 0.9930403402500322
[[7815   22]
 [  86 7595]]
              precision    recall  f1-score   support

         0.0       0.99      1.00      0.99      7837
         1.0       1.00      0.99      0.99      7681

    accuracy                           0.99     15518
   macro avg       0.99      0.99      0.99     15518
weighted avg       0.99      0.99      0.99     15518

